In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EcommerceCaseStudy").getOrCreate()

customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

print("Total Customers:", customers.count())
print("Total Products:", products.count())
print("Total Orders:", orders.count())
print("Total Returned Orders:", returns.count())

Total Customers: 100000
Total Products: 50000


Total Orders: 1000000
Total Returned Orders: 100000


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EcommerceCaseStudy").getOrCreate()

# Load files
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

# Create summary table
summary = [
    ("Customers", customers.count()),
    ("Products", products.count()),
    ("Orders", orders.count()),
    ("Returned Orders", returns.count())
]

result = spark.createDataFrame(summary, ["Metric", "Count"])

result.show(truncate=False)

[Stage 40:>                                                         (0 + 1) / 1]

+---------------+-------+
|Metric         |Count  |
+---------------+-------+
|Customers      |100000 |
|Products       |50000  |
|Orders         |1000000|
|Returned Orders|100000 |
+---------------+-------+



In [4]:
summary = spark.createDataFrame(
    [
        ("Customers", customers.count()),
        ("Products", products.count()),
        ("Orders", orders.count()),
        ("Returned Orders", returns.count())
    ],
    ["metric", "total_count"]
)

summary.show()

+---------------+-----------+
|         metric|total_count|
+---------------+-----------+
|      Customers|     100000|
|       Products|      50000|
|         Orders|    1000000|
|Returned Orders|     100000|
+---------------+-----------+



In [5]:
(summary.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/q1_total_counts"))

In [7]:
products.createOrReplaceTempView("products")
order_items.createOrReplaceTempView("order_items")

category_sales = spark.sql("""
SELECT
    p.category,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
""")

category_sales.show()

(category_sales.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/total_sales_by_category"))

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|        Beauty|7.626693058999963E8|
|Home & Kitchen|7.581388732799902E8|
|         Books|7.464907783499908E8|
|          Toys|7.446190722999846E8|
|   Electronics|7.442665041099958E8|
|        Sports|7.433388681300008E8|
|      Clothing|7.419227945699946E8|
+--------------+-------------------+



In [8]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")

top_customers = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * oi.selling_price) AS total_purchase
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_purchase DESC
LIMIT 10
""")

top_customers.show()

(top_customers.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/top_10_customers"))

+-----------+--------------+------------------+
|customer_id| customer_name|    total_purchase|
+-----------+--------------+------------------+
|      93094|Customer_93094|         181569.68|
|      64560|Customer_64560|169060.39999999997|
|      23289|Customer_23289|161573.80000000002|
|      52275|Customer_52275|153364.78999999998|
|      61218|Customer_61218|         153067.55|
|      52034|Customer_52034|         152680.05|
|      40442|Customer_40442|         151037.32|
|      60528|Customer_60528|         148691.95|
|      84830|Customer_84830|         148363.84|
|      82593|Customer_82593|148281.03999999998|
+-----------+--------------+------------------+



In [9]:
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")

monthly_sales = spark.sql("""
SELECT
    MONTH(o.order_date) AS month,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE YEAR(o.order_date) = (
    SELECT MAX(YEAR(order_date))
    FROM orders
)
GROUP BY MONTH(o.order_date)
ORDER BY month
""")

monthly_sales.show()

(monthly_sales.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/monthly_sales_trend"))

+-----+--------------------+
|month|         total_sales|
+-----+--------------------+
|    1| 4.445777757600032E8|
|    2| 4.153661441999996E8|
|    3| 4.436282454099993E8|
|    4|4.2782097433999574E8|
|    5|4.4481061894999886E8|
|    6| 4.317051540600023E8|
|    7| 4.436705191199994E8|
|    8|4.4109517702000153E8|
|    9| 4.310715260799986E8|
|   10|4.4136378931000113E8|
|   11| 4.336233640400014E8|
|   12| 4.427129083499967E8|
+-----+--------------------+



In [10]:
products.createOrReplaceTempView("products")
order_items.createOrReplaceTempView("order_items")
returns.createOrReplaceTempView("returns")

return_percentage = spark.sql("""
SELECT
    p.category,
    ROUND(
        COUNT(DISTINCT r.order_id) * 100.0 /
        COUNT(DISTINCT oi.order_id), 2
    ) AS return_percentage
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN returns r
    ON oi.order_id = r.order_id
GROUP BY p.category
ORDER BY return_percentage DESC
""")

return_percentage.show()

(return_percentage.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/return_percentage"))

+--------------+-----------------+
|      category|return_percentage|
+--------------+-----------------+
|          Toys|            10.04|
|Home & Kitchen|            10.03|
|        Sports|            10.03|
|   Electronics|            10.02|
|         Books|            10.02|
|        Beauty|            10.02|
|      Clothing|             9.97|
+--------------+-----------------+



In [2]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")

preferred_payment = spark.sql("""
SELECT state,
       payment_mode,
       total_count
FROM (
    SELECT
        c.state,
        o.payment_mode,
        COUNT(*) AS total_count,
        ROW_NUMBER() OVER (
            PARTITION BY c.state
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.state, o.payment_mode
)
WHERE rn = 1
ORDER BY state
""")

preferred_payment.show()

(preferred_payment.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/preferred_payment_mode_by_state"))

+-----+----------------+-----------+
|state|    payment_mode|total_count|
+-----+----------------+-----------+
|   CA|             UPI|      20246|
|   FL|      Debit Card|      20010|
|   GA|     Net Banking|      20041|
|   IL|Cash on Delivery|      20498|
|   MI|      Debit Card|      20416|
|   NC|     Net Banking|      19596|
|   NY|      Debit Card|      20369|
|   OH|     Net Banking|      20351|
|   TX|             UPI|      20065|
|   WA|             UPI|      20244|
+-----+----------------+-----------+



In [5]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")

customer_analysis = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(DISTINCT p.category) AS category_count,
    SUM(oi.quantity * oi.selling_price) AS total_spent
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(DISTINCT p.category) >= 5
   AND SUM(oi.quantity * oi.selling_price) > 100000
ORDER BY total_spent DESC
""")

customer_analysis.show()

(customer_analysis.write
 .mode("overwrite")
 .option("header", "true")
 .csv("output/customers_5_categories_100k"))

NameError: name 'order_items' is not defined